# tutorials.end2end_cli

> Hydra-configured CLI entry points.

In [ ]:
#| default_exp tutorials.end2end_cli


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from pathlib import Path
from typing import Any

import hydra
from omegaconf import DictConfig, OmegaConf

from be_vision_ad_tools.inference.anomaly_score_organizer import predict_and_organize_by_score
from be_vision_ad_tools.inference.unified_inference import unified_inference
from be_vision_ad_tools.inference.unified_with_threshold_posters import unified_inference_with_threshold_posters
from be_vision_ad_tools.training.flexible_trainer import FlexibleTrainingConfig, train_anomaly_model, run_inference_after_training


In [ ]:
#| export
def cfg_to_dict(cfg: DictConfig) -> dict[str, Any]:
    """Convert a Hydra DictConfig to a plain dict with variables resolved."""
    return OmegaConf.to_container(cfg, resolve=True, throw_on_missing=True)  # type: ignore[return-value]


In [ ]:
#| export
def run_train(cfg: DictConfig) -> dict[str, Any]:
    """Train an anomaly-detection model."""
    params = cfg_to_dict(cfg)
    config = FlexibleTrainingConfig(**params)
    return train_anomaly_model(config)


In [ ]:
#| export
def run_infer(cfg: DictConfig) -> dict[str, Any]:
    """Run unified inference on image folder(s) or list file."""
    params = cfg_to_dict(cfg)
    return unified_inference(**params)


In [ ]:
#| export
def run_organize(cfg: DictConfig) -> dict[str, Any]:
    """Predict anomaly scores and organize images into threshold folders."""
    params = cfg_to_dict(cfg)
    return predict_and_organize_by_score(**params)


In [ ]:
#| export
def run_infer_organize(cfg: DictConfig) -> dict[str, Any]:
    """Unified inference + threshold folders + optional posters."""
    params = cfg_to_dict(cfg)
    return unified_inference_with_threshold_posters(**params)


In [ ]:
#| export
def run_train_infer(cfg: DictConfig) -> dict[str, Any]:
    """Train a model, then run inference/posters from training results."""
    root = cfg_to_dict(cfg)
    train_params = root.get('train', root)
    infer_params = root.get('infer_after_training', {})

    training_results = run_train(OmegaConf.create(train_params))
    return run_inference_after_training(training_results, **infer_params)


In [ ]:
#| export
def run_full(cfg: DictConfig) -> dict[str, Any]:
    """Train, then run unified inference with threshold organization."""
    root = cfg_to_dict(cfg)
    train_params = root['train']
    infer_params = root['infer_organize']

    training_results = run_train(OmegaConf.create(train_params))
    if not training_results.get('success', False):
        raise RuntimeError('Training failed; aborting full pipeline')

    export_paths = training_results.get('export_paths', {})
    model_path = export_paths.get('torch') or training_results.get('best_model_path')
    if model_path is None:
        raise RuntimeError('No model path in training results')

    infer_params = dict(infer_params)
    infer_params['model_path'] = str(model_path)
    return run_infer_organize(OmegaConf.create(infer_params))


In [ ]:
#| export
_CONF = str(Path(__file__).resolve().parent / 'conf')


In [ ]:
#| export
@hydra.main(version_base=None, config_path=_CONF, config_name='train')
def train_cli(cfg: DictConfig) -> None:
    """Train an anomaly-detection model."""
    result = run_train(cfg)
    print(result)


In [ ]:
#| export
@hydra.main(version_base=None, config_path=_CONF, config_name='infer')
def infer_cli(cfg: DictConfig) -> None:
    """Score images with unified inference."""
    result = run_infer(cfg)
    print(result)


In [ ]:
#| export
@hydra.main(version_base=None, config_path=_CONF, config_name='organize')
def organize_cli(cfg: DictConfig) -> None:
    """Organize images by anomaly score."""
    result = run_organize(cfg)
    print(result)


In [ ]:
#| export
@hydra.main(version_base=None, config_path=_CONF, config_name='infer_organize')
def infer_organize_cli(cfg: DictConfig) -> None:
    """Inference + threshold folders + posters."""
    result = run_infer_organize(cfg)
    print(result)


In [ ]:
#| export
@hydra.main(version_base=None, config_path=_CONF, config_name='train_infer')
def train_infer_cli(cfg: DictConfig) -> None:
    """Train, then inference/posters from training results."""
    result = run_train_infer(cfg)
    print(result)


In [ ]:
#| export
@hydra.main(version_base=None, config_path=_CONF, config_name='full')
def full_cli(cfg: DictConfig) -> None:
    """Train, then unified inference with threshold organization."""
    result = run_full(cfg)
    print(result)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
